In [ ]:
include("../graph.jl")

using LinearAlgebra
W1  = Variable(randn(64,784), name="l1_weights")
W2  = Variable(randn(10,64), name="l2_weights")
W3 = Variable(randn(1, 10), name="l3_weights")

x = Variable(randn(784), name="x")
y = Variable(randn(1), name="y")

function dense(w, b, x, activation) return activation(w * x .+ b) end
function dense(w, x, activation) return activation(w * x) end
function dense(w, x) return w * x end

function mean_squared_loss(y, ŷ)
    return Constant(0.5) .* (y .- ŷ) .* (y .- ŷ)
end

ReLU(x) = max.(zero.(x), x)

# softmax(x) = exp.(x) ./ sum(exp.(x))

function net(x, w1, w2, w3, y)
    layer1 = dense(w1, x, ReLU)
    layer1.name = "dense1"
    layer2 = dense(w2, layer1, ReLU)
    layer2.name = "dense2"
    layer3 = dense(w3, layer2)
    layer3.name = "dense3"
    L = mean_squared_loss(y, layer3)
    L.name = "loss"

    return create_graph(L)
end

graph = net(x, W1, W2, W3, y);

In [ ]:
losses = Float64[]
lr = 0.01

#Gradient Descent method with constant learning rate
for i in range(1,5)
    loss = forward!(graph)
    backward!(graph)
    W1.outputs -= lr*W1.gradient
    W2.outputs -= lr*W2.gradient
    W3.outputs -= lr*W3.gradient
    push!(losses, loss[1])
    println("Current loss: ", loss[1])

    graph = net(x, W1, W2, W3, y);
end

print(losses)